# Streaming Atlas Data into PyTorch

This notebook shows how to load real Singlet atlas data directly into a **PyTorch** `DataLoader` for model training — no manual `.h5ad` wrangling, no custom collate functions.

`singlet.torch.DataLoader` pulls a sample's `.singlet` bundle from the atlas, decodes the sparse count matrix, and yields batches of dense `float32` tensors shaped `(n_cells, n_genes)` — ready to feed straight into an encoder, autoencoder, or any `torch.nn.Module`.

- **Real data.** Every tensor below comes from a public sample downloaded at runtime; nothing is fabricated.
- **Sparse-friendly.** Counts are stored sparse on disk and densified per batch (or kept sparse via `sparse=True`).
- **GPU-ready.** Pass `device="cuda"` to stream batches directly onto the GPU.
- **Open data.** Atlas samples are reprocessed from public **CC0** archives.

```bash
pip install singlet[torch]
```

In [1]:
import torch
import singlet
from singlet.torch import DataLoader, OnePZDataset

print("singlet", singlet.__version__)
print("torch  ", torch.__version__)

singlet 1.0.0
torch   2.8.0+cu128


## 1. Load a sample as AnnData for context

First, load the sample the ordinary way so we can see exactly what we're streaming. `singlet.load()` downloads the `.singlet` bundle and returns a standard `AnnData`.

In [2]:
adata = singlet.load("GSE149298")
print(adata)
print()
print(f"{adata.n_obs} cells \u00d7 {adata.n_vars} genes")

GSE149298:   0%|          | 0.00/8.13M [00:00<?, ?B/s]

GSE149298: 100%|██████████| 8.13M/8.13M [00:00<00:00, 117MB/s]

AnnData object with n_obs × n_vars = 180 × 38606
    obs: 'gsm_id', 'organism', 'protocol', 'protocol_name', 'sample_source', 'sample_characteristics', 'qc_flag', 'reference_build', 'n_cells_sample'
    var: 'gene_name'
    uns: 'study_meta', 'manifest', 'singlet_bundle_path'

180 cells × 38606 genes


## 2. Build a PyTorch DataLoader

Point the loader at the same accession. It handles download/caching, decoding, optional `log1p` normalization, shuffling, and batching for you.

Each batch is a dense `float32` tensor of shape `(batch_size, n_genes)`.

In [3]:
loader = DataLoader(
    "GSE149298",
    batch_size=64,
    normalize=True,   # log1p-normalized counts
    shuffle=True,
)

print(f"DataLoader ready: {len(loader)} batches of up to 64 cells")

DataLoader ready: 3 batches of up to 64 cells


### Inspect the first few batches

These are real tensors — note the `torch.float32` dtype, the `(n_cells, n_genes)` shape, the device, and the value range (post `log1p`).

In [4]:
for i, batch in enumerate(loader):
    print(f"batch {i}:")
    print(f"  type   {type(batch).__name__}")
    print(f"  shape  {tuple(batch.shape)}")
    print(f"  dtype  {batch.dtype}")
    print(f"  device {batch.device}")
    print(f"  range  [{batch.min().item():.3f}, {batch.max().item():.3f}]")
    if i == 2:
        break

batch 0:
  type   Tensor
  shape  (64, 38606)
  dtype  torch.float32
  device cpu
  range  [0.000, 9.210]


batch 1:
  type   Tensor
  shape  (64, 38606)
  dtype  torch.float32
  device cpu
  range  [0.000, 9.210]


batch 2:
  type   Tensor
  shape  (52, 38606)
  dtype  torch.float32
  device cpu
  range  [0.000, 9.210]


## 3. Feed a batch into a model

Because batches are ordinary `(n_cells, n_genes)` tensors, they drop straight into any `torch.nn.Module`. Here we define a tiny linear encoder and run one batch through it — just to show the tensors flowing into a model. (No training; this is purely illustrative.)

In [5]:
n_genes = adata.n_vars
encoder = torch.nn.Linear(n_genes, 32)

batch = next(iter(loader))
with torch.no_grad():
    z = encoder(batch)

print(f"input  {tuple(batch.shape)}  ({n_genes} genes)")
print(f"latent {tuple(z.shape)}  (32-d embedding per cell)")

input  (64, 38606)  (38606 genes)
latent (64, 32)  (32-d embedding per cell)


## Where to go next

- **Finer control.** Use `OnePZDataset` directly to build your own `torch.utils.data.DataLoader` with custom samplers, collate functions, or multi-epoch logic.
- **GPU training.** Pass `device="cuda"` to `DataLoader` to stream batches straight onto the GPU.
- **Multi-study training.** `source` also accepts a *list* of accessions — e.g. `DataLoader(["GSE149298", "GSE..."], ...)` — to train across many samples with a shared gene space.
- **Sparse batches.** Set `sparse=True` to receive sparse tensors and keep memory low on very wide gene panels.

That's the whole loop: atlas accession in, model-ready `float32` tensors out.